## 0 · The Challenge

> **The mission**: Riverside House — a publishing firm with 7 unpublished novels (~197 chapters, 619k words). Confidentiality clause: no manuscript text goes to any public API. Everything runs on a single laptop CPU. The model is GPT-2 medium (355M parameters).

**What we know so far:**

- GPT-2 medium generates coherent English prose.
- **But it doesn't know Riverside's characters, narrative style, or editorial instructions.**

**What's blocking us:**
Base GPT-2 continues any text, but with no domain knowledge. Ask it about "Chapter 7's antagonist" and it generates something plausible-sounding that is entirely wrong. Three gaps: (1) domain knowledge, (2) instruction following, (3) preference alignment with editorial house style.

**What this chapter unlocks:**
Three fine-tuning techniques layered in order — continued pretraining (domain knowledge), instruction tuning (instruction following), preference-signal training (alignment) — each addressing exactly one of the three gaps.

> **Framework note (TF/Keras):** This is the TensorFlow/Keras version of this notebook series. PyTorch-only tools — LoRA/PEFT, TRL/DPO, bitsandbytes — do not have official Keras equivalents. Each affected section contains an explicit note explaining what is replaced and why, so the pedagogical goals remain fully intact.

# LLM Fine-Tuning Deep Dive, Part 1 of 3: Data-Based Techniques (TensorFlow/Keras)

> **This is Part 1 of a three-notebook fine-tuning arc:**
>
> 1. **Part 1 (this notebook): Data-based techniques** — continued pretraining, instruction tuning, preference-signal fine-tuning.
> 2. [Part 2: Parameter-based techniques + TF quantization](02-llm-finetuning-parameter-techniques.ipynb) — full fine-tuning, partial freezing, TF-native parameter-efficient fine-tuning, TFLite quantization.
> 3. [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) — head-to-head evaluation of all trained checkpoints, held-out perplexity, ablation study, and the final deployment decision.

## The brief: Riverside House needs an in-house AI, not an API call

**Riverside House** is a small publishing firm. Everything under [`content/`](content/) is their **unpublished, proprietary manuscript catalog** — seven complete novels spanning sci-fi, fantasy, mystery, historical fiction, cyberpunk, horror, and literary fiction, still under contract, still unreleased. No manuscript text leaves the building.

Riverside's ask:

1. **An editing assistant** — prompt it with a scene and get back a continuation that actually remembers who Aria Voss is and what the Meridian's Promise is, follows a direct instruction instead of rambling, and reads the way their editors *actually* prefer.
2. **A knowledge base** — marketing, licensing, and new hires who need answers like "who are the six founding families in the mystery novel?" without reading 197 chapters.

**Goal:** fine-tune `gpt2-medium` (~355M parameters) on Riverside's catalog — seven complete novels (~619,000 words / ~3.0 MB, entirely inside [`content/`](content/)) — demonstrating **every major axis of fine-tuning** along the way.

| Step | Concept | Riverside's question | Notebook |
| --- | --- | --- | --- |
| 1 | Continued pretraining | Does it even know our characters and world exist? | Part 1 (this one) |
| 2 | Instruction tuning | Does it follow a "continue this scene" request? | Part 1 (this one) |
| 3 | Preference-signal fine-tuning | Does it write the way our editors actually prefer? | Part 1 (this one) |
| 4 | Full fine-tuning | Best quality — what does it cost? | Part 2 |
| 5 | Partial freezing | A cheaper middle ground — how much quality do we give up? | Part 2 |
| 6 | TF-native parameter-efficient FT | The cheapest option — is it good enough to ship? | Part 2 |
| 7 | Ablation study | What breaks if the deadline forces us to skip a stage? | Part 3 |
| 8 | Head-to-head + held-out perplexity | Which model do we actually deploy in-house? | Part 3 |

### What This Notebook Actually Trains

| # | Training approach | Data format | Checkpoint |
| --- | --- | --- | --- |
| 1 | **Continued pretraining** | Raw novel paragraphs — next-token prediction | [`./checkpoints/non-instruction-full`](../../../checkpoints/non-instruction-full) |
| 2 | **Instruction tuning** | `(instruction + paragraph A, paragraph B)` pairs — with layer freezing | [`./checkpoints/instruction-ft`](../../../checkpoints/instruction-ft) |
| 3 | **Preference-signal fine-tuning** | `(prompt, chosen, rejected)` triples — contrastive reward signal | [`./checkpoints/preference-ft`](../../../checkpoints/preference-ft) |

## Prerequisite Bridge: From Encoder-Decoder Attention to a Decoder-Only Assistant

The transformer foundations introduced three useful shapes: an **encoder** reads an entire input, a **decoder** predicts the next token while respecting a causal mask, and an **encoder-decoder** model lets a decoder attend to an encoded source through cross-attention. Riverside's assistant uses the decoder-only choice.

| Foundation | Role in this chapter | Why Riverside needs it |
| --- | --- | --- |
| Causal decoder | `gpt2-medium` predicts the next token | It can continue prose and answer prompts from one left-to-right context |
| Training objective | Labels say which next tokens should become more likely | Continued pretraining, instruction tuning, and preference-signal FT each change what Riverside teaches the same decoder |
| Encoder / retrieval later | Encodes a query and passages for matching | Finds current, citable manuscript evidence instead of asking the generator to remember every fact |

## Table of Contents (Part 1 of 3)

1. [Why Fine-Tuning? The Three-Gap Problem](#why-fine-tuning-the-three-gap-problem)
2. [Baseline: What Does the Un-Tuned Model Know?](#baseline-what-does-the-un-tuned-model-know)
   - [Loading the Tokenizer](#loading-the-tokenizer)
   - [Loading the Base Model](#loading-the-base-model)
   - [A Reusable `generate()` Helper](#a-reusable-generate-helper)
3. [Concept 1: Continued Pretraining](#concept-1-data-based-non-instructional-fine-tuning-continued-pretraining)
   - [Common Pitfalls: Continued Pretraining](#common-pitfalls-continued-pretraining)
4. [Concept 2: Instruction Tuning](#concept-2-data-based-instructional-supervised-fine-tuning)
   - [TF Note: Layer Freezing Instead of LoRA](#tf-note-layer-freezing-instead-of-lora)
   - [Common Pitfalls: Instruction Tuning](#common-pitfalls-instruction-tuning)
5. [Concept 3: Preference-Signal Fine-Tuning](#concept-3-data-based-preference-alignment)
   - [TF Note: Contrastive Loss Instead of DPO](#tf-note-contrastive-loss-instead-of-dpo)
   - [Common Pitfalls: Preference Fine-Tuning](#common-pitfalls-preference-fine-tuning)
6. [Health Check: All Three Checkpoints](#health-check-all-three-checkpoints)

**Continues in:**

- [Part 2: Parameter-based techniques + TF quantization](02-llm-finetuning-parameter-techniques.ipynb)
- [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb)

---

In [ ]:
# Install required packages (only installs what's missing)
import subprocess, sys
required = [
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('tensorflow', 'tensorflow'),
    ('transformers', 'transformers'),
    ('datasets', 'datasets'),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f'  ok  {pkg}')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  done {pkg}')

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import tensorflow as tf
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
sns.set_theme(style='whitegrid', palette='muted')

MODEL_NAME = 'gpt2-medium'  # ~355M params, real pretrained weights, CPU-trainable
PROMPT = 'Aria Voss stared at the signal counting itself out in prime numbers and'
INSTRUCTION_PREFIX = 'Continue the fiction narrative in the same style:\n\n'

print(f'TensorFlow version: {tf.__version__}')
print(f'Model: {MODEL_NAME}')

In [ ]:
# Resolve the notebook's own directory (content/ lives next to this notebook).
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / 'content'
if not CONTENT_DIR.exists():
    _fallback = Path.cwd() / 'learning' / 'genai' / '04-llm' / 'content'
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f'Content directory: {CONTENT_DIR.absolute()}')

NOVELS = {
    'scifi': 'the-weight-of-distant-light',
    'fantasy': 'the-tidebound-accord',
    'mystery': 'the-cartographers-cipher',
    'historical': 'the-silk-merchants-daughter',
    'cyberpunk': 'neural-drift',
    'horror': 'the-hollow-beneath',
    'literary': 'the-weight-of-tides',
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from the multi-novel corpus for quick CPU demos."""
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob('chapter-*.txt'))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding='utf-8')
            for para in text.split('\n\n'):
                para = para.strip().replace('\n', ' ')
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


sample = load_corpus_paragraphs(novels=['scifi', 'fantasy'], max_chapters=2)
print(f'Loaded {len(sample)} sample paragraphs. First one (truncated):')
print(sample[0][:300], '...')

---

## Why Fine-Tuning? The Three-Gap Problem

A pretrained LLM like GPT-2 has learned language from billions of tokens of web text, giving it strong **general fluency**. But for Riverside's specific job, it has three concrete gaps:

### Gap 1: Domain Knowledge Gap
- **Base model:** asks "Who is Aria Voss?" → makes up something generic
- **After continued pretraining:** "Aria Voss is the Hold systems technician aboard the Meridian's Promise generation ship..."

### Gap 2: Behavior Gap
- **After domain pretraining:** "List the five tides..." → rambles forever instead of answering
- **After instruction tuning:** "The five tides are: water, wind, stone, flame, and void."

### Gap 3: Preference Gap
- **After instruction tuning:** technically correct but verbose/wrong tone
- **After preference-signal FT:** tighter, editorially preferred prose

```
Pretrained base ---> Continued pretraining ---> Instruction tuning ---> Preference-signal FT ---> Production model
    (Gap 0)               (Closes Gap 1)              (Closes Gap 2)            (Closes Gap 3)
```

---

## Baseline: What Does the Un-Tuned Model Know?

Before fine-tuning, let's see what `gpt2-medium` (pretrained on generic web text) produces when prompted with a scenario from Riverside's sci-fi novel. Since it has never seen this story, expect fluent but generic output.

### Loading the Tokenizer

`gpt2-medium` never sees text directly — it's a stack of matrix multiplications that only take numbers. The tokenizer converts text to integer token IDs using byte-pair encoding (BPE). Because GPT-2 was pretrained without padding (continuous fixed-length blocks), we set `pad_token = eos_token` — the standard convention for GPT-family models — so we can batch sequences of different lengths.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Show BPE in action on Riverside's own opening line
example_words = {'noun': 'signal', 'proper noun': 'Aria', 'verb': 'stared'}
for pos, word in example_words.items():
    ids_alone = tokenizer.encode(word)
    ids_mid = tokenizer.encode(' ' + word)
    print(f'{pos.upper()}: {word!r}')
    print(f'  alone:       {tokenizer.convert_ids_to_tokens(ids_alone)}')
    print(f'  mid-sentence: {tokenizer.convert_ids_to_tokens(ids_mid)}')
    print()

### Loading the Base Model

`TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)` loads 355M real pretrained weights into a TF Keras model. This untouched checkpoint is the "before" state every fine-tuning technique is compared against.

The `from_pretrained` call will print a warning about converting from PyTorch weights — this is expected and safe. HuggingFace automatically converts the published PyTorch weights to TF on first use.

In [ ]:
base_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)
total_params = sum(np.prod(v.shape) for v in base_model.trainable_variables)
print(f'Base model loaded: {total_params / 1e6:.1f}M trainable parameters')
print(f'Architecture: {base_model.config.n_layer} transformer blocks, hidden dim {base_model.config.n_embd}')

### A Reusable `generate()` Helper

`generate()` wraps HuggingFace's `model.generate()` to return **only the newly generated tokens** (prompt stripped). Without slicing, `model.generate()` returns `(prompt + continuation)` concatenated — the helper discards the prompt tokens so every `print(generate(...))` call shows just the model's actual output.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Return only the model's continuation, with the prompt stripped."""
    input_ids = tokenizer(prompt, return_tensors='tf')['input_ids']
    prompt_len = input_ids.shape[1]
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
        pad_token_id=tokenizer.pad_token_id,
    )
    decoded = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()
    return decoded if decoded else '[model stopped immediately — sampled EOS as first token]'


print('=== Baseline (no fine-tuning) ===')
print(f'Prompt    : {PROMPT}')
print(f'Completion: {generate(base_model, PROMPT, max_new_tokens=40)}')

### Code Walkthrough: Setup Cells

**What just ran — three building blocks used throughout this entire notebook:**

**1. `tokenizer.pad_token = tokenizer.eos_token`**
GPT-2 was pretrained on fixed-length blocks with no padding. Setting `pad_token = eos_token` (ID 50256) is the standard convention — it tells the tokenizer "treat end-of-sequence as padding" without adding a new row to the embedding matrix.

**2. `TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)`**
Loads the full `gpt2-medium` checkpoint: 355M parameters, 24 transformer blocks, hidden size 1024. HuggingFace's TF model class automatically handles the conversion from published PyTorch weights.

**3. `generate(model, prompt, max_new_tokens=60)` — decoding strategy**

| Parameter | Value | Effect |
| --- | --- | --- |
| `do_sample=True` | Stochastic decoding | Avoids repetitive, deterministic greedy output |
| `top_p=0.9` | Nucleus sampling | Considers only tokens whose cumulative probability ≥ 90% |
| `temperature=0.8` | Slight smoothing | Reduces safe-word dominance without full randomness |

---

## Concept 1 (Data-Based): Non-Instructional Fine-Tuning — Continued Pretraining

**Riverside's question for this section:** Does the model even know our characters and world exist?

**The gap:** The base `gpt2-medium` has never seen Riverside's manuscripts — it was pretrained on generic web text. Ask it about "the Meridian's Promise" or "Aria Voss" and it makes up something plausible but wrong.

**The technique:** Run more pretraining — the exact same next-token-prediction objective the base model was originally trained with — but on Riverside's proprietary catalog instead of generic web text. No new architecture, no special labels: just "continue predicting the next token, but now on our data."

**Data format:** Raw novel paragraphs, no special structure.
**Loss:** Standard causal LM cross-entropy — predict each token given all preceding tokens.
**Parameter strategy:** Full fine-tuning (all weights trainable) — the simplest possible choice, and the right one here since the goal is to absorb vocabulary and style broadly.

**Checkpoint saved to:** `./checkpoints/non-instruction-full`

### Training Objective: Next-Token Prediction

For each sequence of tokens $[t_1, t_2, \ldots, t_T]$, the model learns to predict $t_{i+1}$ given $[t_1, \ldots, t_i]$ for every position $i$. The loss at each position is cross-entropy between the predicted distribution and the true next token. In TF:

```
logits = outputs.logits[:, :-1, :]   # predictions at positions 0..T-2
labels = input_ids[:, 1:]            # targets at positions 1..T-1
loss   = sparse_categorical_crossentropy(labels, logits, from_logits=True)
```

The `[:, :-1]` / `[:, 1:]` shift is how causal LM training works: each position's prediction is scored against the token that actually came next.

In [ ]:
# Tokenize a batch of paragraphs for causal LM training.
# Returns (input_ids, attention_mask) as TF tensors, padded to max_length.
def tokenize_batch(paragraphs, max_length=128):
    enc = tokenizer(
        paragraphs,
        return_tensors='tf',
        max_length=max_length,
        padding='max_length',
        truncation=True,
    )
    return enc['input_ids'], enc['attention_mask']


# Verify tokenization on one paragraph
test_para = load_corpus_paragraphs(novels=['scifi'], max_chapters=1)[0]
test_ids, test_mask = tokenize_batch([test_para])
real_len = int(tf.reduce_sum(test_mask[0]).numpy())
print(f'Tokenized paragraph: {test_ids.shape} → {real_len} real tokens + {128 - real_len} padding')
print(f'First 10 token IDs: {test_ids[0, :10].numpy().tolist()}')
print(f'First 10 tokens:    {tokenizer.convert_ids_to_tokens(test_ids[0, :10].numpy().tolist())}')

In [ ]:
# Continued pretraining: full fine-tuning on the raw Riverside catalog.
# All weights are trainable (no freezing) — this is the simplest parameter strategy.
optimizer_pt = tf.keras.optimizers.Adam(learning_rate=5e-5)
# A fresh copy of gpt2-medium: full fine-tuning should not start from a previously modified model
pretrain_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)


@tf.function
def pretrain_step(input_ids, attention_mask):
    """One gradient update. Labels = input_ids shifted left by 1 (causal LM objective)."""
    labels = input_ids[:, 1:]   # targets: tokens at positions 1..T
    with tf.GradientTape() as tape:
        outputs = pretrain_model(input_ids, attention_mask=attention_mask, training=True)
        logits = outputs.logits[:, :-1, :]  # predictions at positions 0..T-1
        # Mask out padding positions: only compute loss on real tokens
        pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)  # shape (B, T-1)
        token_loss = tf.keras.losses.sparse_categorical_crossentropy(
            labels, logits, from_logits=True
        )  # shape (B, T-1)
        loss = tf.reduce_sum(token_loss * pad_mask) / (tf.reduce_sum(pad_mask) + 1e-9)
    grads = tape.gradient(loss, pretrain_model.trainable_variables)
    optimizer_pt.apply_gradients(zip(grads, pretrain_model.trainable_variables))
    return loss


print('Continued pretraining setup complete.')

In [ ]:
# Build dataset and run the continued pretraining loop.
# Loads 4 genres, first 4 chapters each — fast CPU demo subset.
pt_paragraphs = load_corpus_paragraphs(
    novels=['scifi', 'fantasy', 'mystery', 'historical'], max_chapters=4
)
print(f'Corpus: {len(pt_paragraphs)} paragraphs loaded for continued pretraining')

BATCH_SIZE = 4
MAX_STEPS = 60
LOG_EVERY = 10
loss_history_pt = []

for step in range(MAX_STEPS):
    batch_start = (step * BATCH_SIZE) % max(1, len(pt_paragraphs) - BATCH_SIZE)
    batch = pt_paragraphs[batch_start : batch_start + BATCH_SIZE]
    if not batch:
        continue
    ids, mask = tokenize_batch(batch)
    loss_val = pretrain_step(ids, mask)
    loss_history_pt.append(float(loss_val))
    if (step + 1) % LOG_EVERY == 0:
        print(f'  Step {step + 1:3d}/{MAX_STEPS}  loss={float(loss_val):.4f}')

print(f'\nFinal loss: {loss_history_pt[-1]:.4f}  (started at {loss_history_pt[0]:.4f})')

In [ ]:
# Save checkpoint and test the pre-trained model
pretrain_model.save_pretrained('./checkpoints/non-instruction-full')
print('Saved: ./checkpoints/non-instruction-full')

# Before/after comparison
print('\n=== Before continued pretraining (base model) ===')
print(f'Completion: {generate(base_model, PROMPT, max_new_tokens=40)}')

print('\n=== After continued pretraining (non-instruction-full) ===')
print(f'Completion: {generate(pretrain_model, PROMPT, max_new_tokens=40)}')

# Plot the training loss curve
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(loss_history_pt, color='steelblue', linewidth=2, label='Training loss')
ax.set_xlabel('Training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Concept 1: Continued Pretraining — Loss Curve (full fine-tuning)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Code Walkthrough: Continued Pretraining

**What `pretrain_step()` does — three operations:**

**1. The causal LM shift**
```
labels  = input_ids[:, 1:]       # every token except the first
logits  = outputs.logits[:, :-1] # every prediction except the last
```
Position `i`'s prediction is graded against label `i+1`. This one-slot offset is the entire causal LM objective: *predict the next token*.

**2. Padding mask**
GPT-2's tokenizer pads shorter paragraphs to `max_length=128`. Without masking, the model would compute loss on the padding tokens too — teaching it to predict EOS tokens, not prose. `pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)` zeros out the loss at every padding position, so only real manuscript tokens count.

**3. Weight update via `GradientTape`**
TF's `GradientTape` is the Keras-native equivalent of PyTorch's `loss.backward()`. Inside the `with` block, every operation is recorded; `tape.gradient(loss, trainable_variables)` computes $\partial L / \partial W$ for every parameter simultaneously (chain rule, in reverse order). `optimizer.apply_gradients` then nudges each weight by $-lr \times \nabla W$.

### Common Pitfalls: Continued Pretraining

| Pitfall | What happens | Fix |
| --- | --- | --- |
| Too few steps | Model output barely changes | Increase `MAX_STEPS` or use more data |
| Catastrophic forgetting | Model forgets general English | Use a small learning rate (`5e-5`), limit steps |
| Not masking padding | Model learns to predict EOS repetitively | Always apply `pad_mask` to the loss |
| Evaluating without `.generate()` | `outputs.logits` ≠ final continuation | Use the `generate()` helper, not raw logits |

---

## Concept 2 (Data-Based): Instructional Supervised Fine-Tuning

**Riverside's question for this section:** after domain pretraining, the model knows the lore — but it still *rambles*. Ask it to "list the five tides" and it starts a scholarly debate instead of answering. How do we teach it to follow a direct instruction?

**The technique:** Supervised fine-tuning (SFT) on `(instruction, completion)` pairs. The prompt format wraps each paragraph with an explicit instruction prefix; the model learns to produce the continuation when given that structure.

**Data format:** `instruction_prefix + paragraph_A → paragraph_B`  
**Loss:** Same causal LM cross-entropy, but only over the *completion* tokens (not the instruction prefix).
**Checkpoint saved to:** `./checkpoints/instruction-ft`

### TF Note: Layer Freezing Instead of LoRA

> **Note: LoRA/PEFT is PyTorch-only.** The PyTorch version of this notebook uses `peft.get_peft_model()` with a `LoraConfig` to inject trainable low-rank matrices while keeping the base model frozen. PEFT does not have an official TensorFlow equivalent.
>
> **This Keras version demonstrates the same concept using TF-native layer freezing:**
>
| PyTorch (LoRA) | TensorFlow / Keras (layer freezing) |
| --- | --- |
| Freeze 100% of base weights | Freeze first 18/24 transformer blocks |
| Add small trainable A/B matrices (<1% params) | Leave last 6 blocks + output head trainable (~21% params) |
| `peft.get_peft_model(model, lora_config)` | `for layer in model.transformer.h[:18]: layer.trainable = False` |
| Swappable adapters on the same frozen base | Multiple checkpoints (one per use case) |
>
> The pedagogical goal is identical: **teach instruction-following while minimising how much of the model changes**, so the general language ability learned in pretraining is preserved. Layer freezing achieves this by keeping the early (general) transformer blocks intact and only updating the task-specific final blocks.
>
> Full LoRA requires training a model architecture that has the LoRA adapter matrices injected into it. To replicate exact LoRA behaviour in TF, you would need to subclass the attention layers — out of scope here. If you need true LoRA in TF, see the `keras-nlp` project or use the PyTorch version of this notebook.

In [ ]:
# Build instruction-following training pairs from the corpus.
# Format: (instruction_prefix + paragraph_A, paragraph_B) where A leads into B.
def build_instruction_pairs(novels, max_chapters=4):
    """Create (prompt, completion) pairs for supervised fine-tuning."""
    pairs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob('chapter-*.txt'))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding='utf-8')
            paras = [p.strip().replace('\n', ' ') for p in text.split('\n\n') if len(p.strip()) >= 200]
            for i in range(len(paras) - 1):
                prompt = INSTRUCTION_PREFIX + paras[i]
                completion = paras[i + 1]
                pairs.append({'prompt': prompt, 'completion': completion})
    return pairs


instruct_pairs = build_instruction_pairs(
    novels=['scifi', 'cyberpunk', 'literary'], max_chapters=3
)
print(f'Instruction pairs built: {len(instruct_pairs)}')
if instruct_pairs:
    print(f'\nSample prompt  (truncated): {instruct_pairs[0]["prompt"][:150]}...')
    print(f'Sample completion (truncated): {instruct_pairs[0]["completion"][:100]}...')

In [ ]:
# Load a fresh gpt2-medium base and apply layer freezing.
# Freeze the first (n_layers - 6) transformer blocks; leave the last 6 + output head trainable.
instruct_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)

n_layers = instruct_model.config.n_layer  # 24 for gpt2-medium
unfreeze_from = n_layers - max(2, n_layers // 4)  # freeze all but the last ~25%

# Freeze early transformer blocks (general language features)
for i, block in enumerate(instruct_model.transformer.h):
    if i < unfreeze_from:
        block.trainable = False

# Keep embeddings and early layers frozen too
instruct_model.transformer.wte.trainable = False
instruct_model.transformer.wpe.trainable = False

# Count trainable parameters
trainable_params = sum(np.prod(v.shape) for v in instruct_model.trainable_variables)
total_params_cnt = sum(np.prod(v.shape) for v in instruct_model.variables)
print(f'Layer freezing applied: {n_layers} blocks, unfreezing from block {unfreeze_from}')
print(f'Trainable: {trainable_params:,} / {total_params_cnt:,} ({trainable_params / total_params_cnt * 100:.1f}%)')
print(f'Frozen:    {total_params_cnt - trainable_params:,} / {total_params_cnt:,} ({(total_params_cnt - trainable_params) / total_params_cnt * 100:.1f}%)')

In [ ]:
# Instruction tuning train step.
# The prompt tokens are included in the input but their loss contribution can be masked;
# for simplicity here we train on the full concatenated (prompt + completion) sequence,
# which is the standard supervised causal LM approach.
optimizer_instruct = tf.keras.optimizers.Adam(learning_rate=2e-4)


@tf.function
def instruct_step(input_ids, attention_mask):
    labels = input_ids[:, 1:]
    with tf.GradientTape() as tape:
        outputs = instruct_model(input_ids, attention_mask=attention_mask, training=True)
        logits = outputs.logits[:, :-1, :]
        pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)
        token_loss = tf.keras.losses.sparse_categorical_crossentropy(
            labels, logits, from_logits=True
        )
        loss = tf.reduce_sum(token_loss * pad_mask) / (tf.reduce_sum(pad_mask) + 1e-9)
    # Only update trainable_variables (frozen layers have no variables in this list)
    grads = tape.gradient(loss, instruct_model.trainable_variables)
    optimizer_instruct.apply_gradients(zip(grads, instruct_model.trainable_variables))
    return loss


# Build tokenized dataset from instruction pairs
instruct_texts = [
    p['prompt'] + '\n\n' + p['completion'] for p in instruct_pairs
]
MAX_STEPS_INSTRUCT = 60
loss_history_instruct = []

for step in range(MAX_STEPS_INSTRUCT):
    idx = (step * BATCH_SIZE) % max(1, len(instruct_texts) - BATCH_SIZE)
    batch = instruct_texts[idx : idx + BATCH_SIZE]
    if not batch:
        continue
    ids, mask = tokenize_batch(batch)
    loss_val = instruct_step(ids, mask)
    loss_history_instruct.append(float(loss_val))
    if (step + 1) % LOG_EVERY == 0:
        print(f'  Step {step + 1:3d}/{MAX_STEPS_INSTRUCT}  loss={float(loss_val):.4f}')

print(f'\nFinal loss: {loss_history_instruct[-1]:.4f}')

In [ ]:
# Save instruction-tuned checkpoint and test
instruct_model.save_pretrained('./checkpoints/instruction-ft')
print('Saved: ./checkpoints/instruction-ft')

instruct_prompt = INSTRUCTION_PREFIX + PROMPT + '\n\n'
print('\n=== Instruction-tuned (layer-frozen) ===')
print(f'Prompt: {INSTRUCTION_PREFIX!r} + (sci-fi opening)')
print(f'Completion: {generate(instruct_model, instruct_prompt, max_new_tokens=60)}')

# Test without the instruction prefix (should perform worse — expected)
print('\n=== Without instruction prefix (should ramble — shows prefix matters) ===')
print(f'Completion: {generate(instruct_model, PROMPT, max_new_tokens=40)}')

### Code Walkthrough: Instruction Tuning with Layer Freezing

**Three things to understand about this training setup:**

**1. Why freeze early layers?**
In transformer models, early blocks learn general language features (syntax, common vocabulary, English grammar) while later blocks learn task-specific patterns. By freezing the first 18/24 blocks, we preserve GPT-2's general language ability while only updating the task-specific late blocks. The risk of catastrophic forgetting — the model "forgetting" how to write fluent English while it learns Riverside's instruction format — is drastically reduced.

**2. `instruct_model.transformer.h[i].trainable = False`**
In TF/Keras, setting `layer.trainable = False` tells the layer (and all its sub-layers) to stop accumulating gradients. The `model.trainable_variables` list automatically excludes frozen layers, so `optimizer.apply_gradients` never touches them.

**3. Without the instruction prefix (test cell above)**
The instruction-tuned model was trained on sequences beginning with `INSTRUCTION_PREFIX`. Without it, the model sees a different distribution than it was trained on — expect degraded output. This is the expected behaviour for an instruction-tuned model: it's specialised for the prompt format, not general continuation.

### Common Pitfalls: Instruction Tuning

| Pitfall | What happens | Fix |
| --- | --- | --- |
| Wrong prompt format at inference | Degraded, off-format output | Always use the exact prefix from training |
| Too many layers frozen | Model can't learn the instruction format | Unfreeze at least last 25% of blocks + output head |
| Too few pairs | Model memorises instead of generalising | Use diverse (instruction, completion) pairs across genres |
| Forgetting to reset `trainable` | All layers update, nullifying freezing | Set `trainable=False` before calling `trainable_variables` |

---

## Concept 3 (Data-Based): Preference Alignment

**Riverside's question for this section:** even with instruction-following, some outputs are technically correct but stylistically wrong — too verbose, wrong tone, unhelpful focus. How do we teach the model to prefer the editorial style Riverside's editors actually like?

**The technique:** Preference-signal fine-tuning. Given `(prompt, chosen_continuation, rejected_continuation)` triples, train the model to make the "chosen" response more probable than the "rejected" one.

**Data format:** `(prompt, chosen, rejected)` triples.
**Loss:** Contrastive reward signal — see note below.
**Checkpoint saved to:** `./checkpoints/preference-ft`

### TF Note: Contrastive Loss Instead of DPO

> **Note: DPO/TRL is PyTorch-only.** The PyTorch version of this notebook uses `trl.DPOTrainer` with a reference model and the full DPO loss (including KL penalty). TRL does not have an official TensorFlow equivalent.
>
> **This Keras version demonstrates the same concept using a simplified contrastive reward-signal loss:**
>
> | PyTorch (DPO via TRL) | TensorFlow / Keras (contrastive reward signal) |
> | --- | --- |
> | Full DPO: $-\log \sigma(\beta(\log \frac{\pi_{\theta}(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_{\theta}(y_l|x)}{\pi_{ref}(y_l|x)}))$ | Simplified: $-\log \sigma(\log P_{chosen} - \log P_{rejected})$ |
> | Requires a frozen reference model | No reference model needed |
> | `trl.DPOTrainer` handles the full recipe | Manual `GradientTape` with contrastive loss |
>
> The simplified version omits the KL-divergence term that anchors the model to a reference policy. This makes it less stable for large datasets (it can over-fit to the preference signal) but correct and runnable on CPU without the TRL dependency. The key intuition is preserved: **the model learns to assign higher probability to the preferred completion than the rejected one**.
>
> For production TF preference alignment, consider: (1) RLHF via a reward model (possible in pure TF but complex), (2) using this notebook's PyTorch counterpart with a GPU, or (3) the Keras Team's planned alignment utilities.

In [ ]:
# Preference pairs for Riverside House.
# Each triple: (prompt, chosen_continuation, rejected_continuation).
# "chosen" = preferred editorial style (tight, characterful, stops when done)
# "rejected" = acceptable but suboptimal (verbose, generic, overlong)
PREFERENCE_PAIRS = [
    {
        'prompt': INSTRUCTION_PREFIX + 'Aria Voss checked the Meridian and',
        'chosen': ' found node seventeen humming three tones above baseline — she knew that sound.',
        'rejected': ' found that everything was working fine and the systems were all in order and there were no problems.',
    },
    {
        'prompt': INSTRUCTION_PREFIX + 'The signal repeated itself and',
        'chosen': ' Aria\'s stomach dropped. Prime numbers. Someone — or something — was counting.',
        'rejected': ' then it repeated again, and again, and again, continuing to repeat in the same way it always had.',
    },
    {
        'prompt': INSTRUCTION_PREFIX + 'Captain Reyes called the crew to the bridge and',
        'chosen': ' said nothing for a long moment. The silence was its own kind of answer.',
        'rejected': ' then he began to speak to them about many different things that were happening on the ship and in space.',
    },
    {
        'prompt': INSTRUCTION_PREFIX + 'The Tidebound Accord was broken when',
        'chosen': ' Kael touched the void stone — and the sea answered in a voice made of drowned bells.',
        'rejected': ' something happened that was not supposed to happen according to the rules of the Accord that everyone knew about.',
    },
    {
        'prompt': INSTRUCTION_PREFIX + 'The detective found the cipher and',
        'chosen': ' understood at once why the cartographer had hidden it. The map was not the treasure. The map was the warning.',
        'rejected': ' then proceeded to examine it carefully to figure out what it meant and what information it might contain.',
    },
]
print(f'Preference pairs: {len(PREFERENCE_PAIRS)}')
print(f'\nSample chosen  : {PREFERENCE_PAIRS[0]["chosen"]}')
print(f'Sample rejected: {PREFERENCE_PAIRS[0]["rejected"]}')

In [ ]:
# Load the instruction-tuned model as the starting point for preference fine-tuning.
# (DPO builds on top of an instruction-tuned base — same convention here.)
policy_model = TFGPT2LMHeadModel.from_pretrained('./checkpoints/instruction-ft')
optimizer_pref = tf.keras.optimizers.Adam(learning_rate=1e-5)  # small LR for preference stage


def compute_sequence_log_prob(model, input_ids, attention_mask):
    """Compute the mean log probability of a sequence (negative cross-entropy)."""
    labels = input_ids[:, 1:]
    outputs = model(input_ids, attention_mask=attention_mask, training=False)
    logits = outputs.logits[:, :-1, :]
    pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)
    token_loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels, logits, from_logits=True
    )
    # Mean cross-entropy over real tokens = -log P(sequence) / T
    mean_ce = tf.reduce_sum(token_loss * pad_mask) / (tf.reduce_sum(pad_mask) + 1e-9)
    return -mean_ce  # log probability (higher = more likely)


def preference_train_step(chosen_text, rejected_text):
    """One contrastive preference update step."""
    chosen_ids, chosen_mask = tokenize_batch([chosen_text])
    rejected_ids, rejected_mask = tokenize_batch([rejected_text])
    with tf.GradientTape() as tape:
        log_prob_chosen = compute_sequence_log_prob(policy_model, chosen_ids, chosen_mask)
        log_prob_rejected = compute_sequence_log_prob(policy_model, rejected_ids, rejected_mask)
        # Contrastive loss: push chosen above rejected
        # -log(sigmoid(log_P_chosen - log_P_rejected))
        margin = log_prob_chosen - log_prob_rejected
        pref_loss = -tf.math.log(tf.sigmoid(margin) + 1e-8)
    grads = tape.gradient(pref_loss, policy_model.trainable_variables)
    optimizer_pref.apply_gradients(zip(grads, policy_model.trainable_variables))
    return float(pref_loss), float(margin)


# Training loop: cycle over preference pairs
MAX_STEPS_PREF = 30
loss_history_pref = []
margin_history = []

for step in range(MAX_STEPS_PREF):
    pair = PREFERENCE_PAIRS[step % len(PREFERENCE_PAIRS)]
    chosen_full = pair['prompt'] + pair['chosen']
    rejected_full = pair['prompt'] + pair['rejected']
    pref_loss, margin = preference_train_step(chosen_full, rejected_full)
    loss_history_pref.append(pref_loss)
    margin_history.append(margin)
    if (step + 1) % 10 == 0:
        print(f'  Step {step + 1:3d}/{MAX_STEPS_PREF}  pref_loss={pref_loss:.4f}  margin={margin:.4f}')

print(f'\nFinal margin: {margin_history[-1]:.4f}  (positive = model prefers chosen over rejected)')

In [ ]:
# Save preference-aligned checkpoint and test
policy_model.save_pretrained('./checkpoints/preference-ft')
print('Saved: ./checkpoints/preference-ft')

# Compare instruction-tuned vs preference-aligned on a preference test prompt
pref_test_prompt = INSTRUCTION_PREFIX + 'Aria Voss checked the Meridian and'
print('\n=== Instruction-tuned (before preference FT) ===')
print(f'Completion: {generate(instruct_model, pref_test_prompt, max_new_tokens=50)}')
print('\n=== Preference-aligned (after contrastive training) ===')
print(f'Completion: {generate(policy_model, pref_test_prompt, max_new_tokens=50)}')

# Plot margin over training
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ax1.plot(loss_history_pref, color='darkorange', linewidth=2)
ax1.set_xlabel('Step'); ax1.set_ylabel('Preference loss'); ax1.set_title('Preference Loss (lower = better)', fontweight='bold'); ax1.grid(alpha=0.3)
ax2.plot(margin_history, color='mediumseagreen', linewidth=2)
ax2.axhline(0, color='red', linestyle='--', alpha=0.5, label='margin=0 (indifferent)')
ax2.set_xlabel('Step'); ax2.set_ylabel('log P(chosen) - log P(rejected)'); ax2.set_title('Preference Margin (higher = model prefers chosen)', fontweight='bold'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Code Walkthrough: Preference-Signal Fine-Tuning

**The contrastive reward signal — three components:**

**1. `compute_sequence_log_prob()`**
Returns the mean log probability of a sequence: $\log P(\text{sequence}) \approx -\frac{1}{T} \sum_i CE(\text{logit}_i, \text{label}_i)$. A lower cross-entropy loss means the model assigns higher probability to that sequence.

**2. The contrastive loss**
```python
margin = log_P_chosen - log_P_rejected
loss   = -log(sigmoid(margin))
```
When `margin > 0`, the model already prefers the chosen sequence — `sigmoid(margin) > 0.5`, so `loss < log(2)`. The optimizer drives `margin` higher (toward $+\infty$) to minimise this loss. The full DPO objective adds a KL penalty against a frozen reference model, which prevents the margin from growing so large the model diverges from its instruction-following behaviour — the simplified version here omits that guardrail.

**3. Starting from the instruction-tuned checkpoint**
DPO (and our simplified version) is always applied *on top of* an instruction-tuned model, never directly on the base. The model needs to already know how to follow the instruction format before it can learn which responses within that format are preferred.

### Common Pitfalls: Preference Fine-Tuning

| Pitfall | What happens | Fix |
| --- | --- | --- |
| Starting from base (not instruction-tuned) | Preference signal is too noisy to converge | Always start from the instruction-tuned checkpoint |
| Too few preference pairs | Model memorises pairs, doesn't generalise | 30+ diverse pairs minimum; 100+ for production |
| High learning rate | Model forgets instruction-following behaviour | Keep LR small (1e-5 or lower) for the preference stage |
| Margin doesn't increase | Chosen and rejected are too similar | Write pairs with clearer stylistic contrast |

---

## Health Check: All Three Checkpoints

Parts 2 and 3 reload these checkpoints from disk rather than assuming this kernel is still running. Confirming all three exist on disk before continuing.

In [ ]:
import os

checkpoints = [
    './checkpoints/non-instruction-full',
    './checkpoints/instruction-ft',
    './checkpoints/preference-ft',
]

print('Health check: checkpoint directories')
print('=' * 60)
all_ok = True
for ckpt in checkpoints:
    exists = os.path.isdir(ckpt)
    files = os.listdir(ckpt) if exists else []
    has_weights = any('model' in f or 'tf_model' in f or 'weights' in f or '.h5' in f or 'saved_model' in f for f in files)
    status = 'OK' if (exists and len(files) > 0) else 'MISSING'
    if status != 'OK':
        all_ok = False
    print(f'  {status}  {ckpt}  ({len(files)} files)')

print('=' * 60)
if all_ok:
    print('All three checkpoints exist. Part 2 can reload from disk.')
else:
    print('WARNING: one or more checkpoints missing — re-run the training cells above.')

# Final before/after comparison on all three models
print('\n=== Final comparison on the test prompt ===')
print(f'Prompt: {PROMPT}')
print(f'\nBase (no FT)     : {generate(base_model, PROMPT, max_new_tokens=40)}')
print(f'Non-instruction  : {generate(pretrain_model, PROMPT, max_new_tokens=40)}')
print(f'Instruction-ft   : {generate(instruct_model, INSTRUCTION_PREFIX + PROMPT + chr(10) * 2, max_new_tokens=40)}')
print(f'Preference-ft    : {generate(policy_model, INSTRUCTION_PREFIX + PROMPT + chr(10) * 2, max_new_tokens=40)}')

---

## Summary: What Part 1 Built

Three distinct models are now saved to disk, each answering one of Riverside's three questions:

| # | Checkpoint | Answers | Technique | TF note |
| --- | --- | --- | --- | --- |
| 1 | `non-instruction-full` | Does it know our world? | Continued pretraining, full FT | No change from PyTorch |
| 2 | `instruction-ft` | Does it follow instructions? | Instruction tuning + **layer freezing** | Replaces LoRA/PEFT |
| 3 | `preference-ft` | Does it write our way? | **Contrastive reward-signal FT** | Replaces TRL/DPO |

**What Part 2 covers:** The same continued-pretraining objective is used as a controlled test bed to compare the *parameter axis* directly: full fine-tuning vs. partial freezing vs. TF-native parameter-efficient fine-tuning. Everything trained on the same data — only how many weights move changes.

**Key insights from Part 1:**

- Continued pretraining shifts the model's probability distribution toward domain vocabulary without changing its architecture.
- Layer freezing (the TF substitute for LoRA) achieves instruction tuning with ~21% of parameters — the frozen early blocks preserve general language ability while the trainable late blocks absorb the instruction format.
- The preference contrastive loss is the same mathematical operation as DPO's core term: $-\log \sigma(\log P_{chosen} - \log P_{rejected})$. The DPO paper's additional KL term stabilises training at scale; the simplified version here is sufficient for demonstration.
- All three data-based stages are **additive**: continued pretraining → instruction tuning → preference fine-tuning is the correct order. Skipping stages or reversing order degrades results (demonstrated in Part 3's ablation study).